# Day 3 v3: ML Optimization — Vietnamese Price Prediction (<= 1M VND)**Target:** RMSLE from 0.58 to <= 0.45**Improvements:** Log-transform, Category features, Arch C (char_wb), Ridge, LightGBM Optuna, Blending

In [ ]:
!nvidia-smi

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))

import random
import time
import pickle

import numpy as np
from tqdm.auto import tqdm
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate, rmsle

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
PRICE_THRESHOLD = 1_000_000
CACHE_DIR = Path(".")
EVAL_SIZE = "all"

random.seed(SEED)
np.random.seed(SEED)

## 1. Load Data + Filter

In [ ]:
train, val, test = Item.from_hub(DATASET)
print(f"Raw: {len(train):,} train | {len(val):,} val | {len(test):,} test")

train = [item for item in train if item.price <= PRICE_THRESHOLD]
val = [item for item in val if item.price <= PRICE_THRESHOLD]
test = [item for item in test if item.price <= PRICE_THRESHOLD]
print(f"Filtered <= {PRICE_THRESHOLD:,} VND:")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

train_prices = [item.price for item in train]
val_prices = [item.price for item in val]
prices = np.array(train_prices, dtype=float)
documents = [item.summary for item in train]
categories_train = [[item.category] for item in train]

log_prices = np.log1p(prices)
print(f"  Price: {prices.min():,.0f} - {prices.max():,.0f} VND | Mean: {prices.mean():,.0f} | Median: {np.median(prices):,.0f}")
print(f"  Log: {log_prices.min():.2f} - {log_prices.max():.2f} | Mean: {log_prices.mean():.2f}")

## 2. Tokenization (cache)

In [ ]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

cache_train = CACHE_DIR / "tokenized_train_1m.pkl"
cache_test = CACHE_DIR / "tokenized_test_1m.pkl"
cache_val = CACHE_DIR / "tokenized_val_1m.pkl"

def load_or_tokenize(cache_path, texts, desc):
    if cache_path.exists():
        print(f"Loading: {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    t0 = time.time()
    print(f"Tokenizing {len(texts):,} docs...")
    with Pool(4) as p:
        result = list(tqdm(p.imap(tokenize_one, texts, chunksize=500), total=len(texts), desc=desc))
    print(f"  Done: {time.time()-t0:.1f}s")
    with open(cache_path, "wb") as f:
        pickle.dump(result, f)
    return result

tokenized_train = load_or_tokenize(cache_train, documents, "train")
tokenized_test = load_or_tokenize(cache_test, [item.summary for item in test], "test")
tokenized_val = load_or_tokenize(cache_val, [item.summary for item in val], "val")
tokenized_test_map = {item.summary: tok for item, tok in zip(test, tokenized_test)}

print(f"Sample: {tokenized_train[0][:100]}")

## 3. Feature Engineering

In [ ]:
# Arch B: word TF-IDF
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10000)
X_text_train = vectorizer_b.fit_transform(tokenized_train)
X_text_test = vectorizer_b.transform(tokenized_test)
X_text_val = vectorizer_b.transform(tokenized_val)
print(f"Arch B: {X_text_train.shape} ({time.time()-t0:.1f}s)")

# Arch C: word + char_wb
t0 = time.time()
arch_c = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=5000)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5000)),
])
X_text_c_train = arch_c.fit_transform(tokenized_train)
X_text_c_test = arch_c.transform(tokenized_test)
X_text_c_val = arch_c.transform(tokenized_val)
print(f"Arch C: {X_text_c_train.shape} ({time.time()-t0:.1f}s)")

# Category one-hot
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat_train = cat_encoder.fit_transform(categories_train)
X_cat_test = cat_encoder.transform([[item.category] for item in test])
X_cat_val = cat_encoder.transform([[item.category] for item in val])
print(f"Categories: {list(cat_encoder.categories_[0])}")

# Combined
X_bc_train = hstack([X_text_train, X_cat_train])
X_bc_test = hstack([X_text_test, X_cat_test])
X_bc_val = hstack([X_text_val, X_cat_val])
X_cc_train = hstack([X_text_c_train, X_cat_train])
X_cc_test = hstack([X_text_c_test, X_cat_test])
X_cc_val = hstack([X_text_c_val, X_cat_val])
print(f"B+Cat: {X_bc_train.shape} | C+Cat: {X_cc_train.shape}")

In [ ]:
# Helper: evaluate log-transform model
def evaluate_log_model(model, X_test_matrix, test_data, title):
    pred_log = model.predict(X_test_matrix)
    pred_price = np.clip(np.expm1(pred_log), 0, None)
    y_true = np.array([item.price for item in test_data], dtype=float)
    rmsle_val = rmsle(y_true, pred_price)
    mae_val = float(np.mean(np.abs(y_true - pred_price)))
    mask = y_true > 0
    mape_val = float(np.mean(np.abs((y_true[mask] - pred_price[mask]) / y_true[mask])) * 100)
    from sklearn.metrics import r2_score
    r2_val = r2_score(y_true, pred_price) * 100
    print(f"
{title} ({len(test_data)} items):")
    print(f"  RMSLE: {rmsle_val:.4f} | MAE: {mae_val:,.0f} | MAPE: {mape_val:.1f}% | R2: {r2_val:.1f}%")
    return {"rmsle": rmsle_val, "mae": mae_val, "mape": mape_val, "r2": r2_val}

results = {}

---## Phase 1: Log-transform — All Models

In [ ]:
# Median baseline
training_median = float(np.median(prices))
def median_pricer(item):
    return training_median
results["Median (baseline)"] = evaluate(median_pricer, test, size=EVAL_SIZE)

In [ ]:
# Ridge (B+Cat, log)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_bc_train, log_prices)
results["Ridge (B+Cat, log)"] = evaluate_log_model(ridge_model, X_bc_test, test, "Ridge (B+Cat, log)")

In [ ]:
# Ridge (C+Cat, log)
ridge_c = Ridge(alpha=1.0)
ridge_c.fit(X_cc_train, log_prices)
results["Ridge (C+Cat, log)"] = evaluate_log_model(ridge_c, X_cc_test, test, "Ridge (C+Cat, log)")

In [ ]:
# LR (B+Cat, log) for comparison
lr_log = LinearRegression()
lr_log.fit(X_bc_train, log_prices)
results["LR (B+Cat, log)"] = evaluate_log_model(lr_log, X_bc_test, test, "LR (B+Cat, log)")

In [ ]:
# Random Forest (B+Cat, log, subset 40K)
SUBSET_RF = 40_000
t0 = time.time()
rf_log = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=6, verbose=1)
rf_log.fit(X_bc_train[:SUBSET_RF], log_prices[:SUBSET_RF])
print(f"
RF train: {time.time()-t0:.1f}s")
results["RF (B+Cat, log)"] = evaluate_log_model(rf_log, X_bc_test, test, "RF (B+Cat, log)")

In [ ]:
# XGBoost (B+Cat, log)
t0 = time.time()
pbar_xgb = tqdm(total=1000, desc="XGBoost")
class XGBProgress(xgb.callback.TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        pbar_xgb.update(1)
        return False
    def after_training(self, model):
        pbar_xgb.close()
        return model
xgb_log = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6,
    tree_method="hist", callbacks=[XGBProgress()],
)
xgb_log.fit(X_bc_train, log_prices)
print(f"XGBoost train: {time.time()-t0:.1f}s")
results["XGBoost (B+Cat, log)"] = evaluate_log_model(xgb_log, X_bc_test, test, "XGBoost (B+Cat, log)")

In [ ]:
# LightGBM (B+Cat, log)
t0 = time.time()
pbar_lgb = tqdm(total=1000, desc="LightGBM")
def lgb_cb(env): pbar_lgb.update(1)
lgb_log = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6, verbose=-1)
lgb_log.fit(X_bc_train, log_prices, callbacks=[lgb_cb])
pbar_lgb.close()
print(f"LightGBM train: {time.time()-t0:.1f}s")
results["LightGBM (B+Cat, log)"] = evaluate_log_model(lgb_log, X_bc_test, test, "LightGBM (B+Cat, log)")

In [ ]:
# CatBoost (B+Cat, log)
t0 = time.time()
cb_log = CatBoostRegressor(iterations=1000, learning_rate=0.1, random_seed=SEED, verbose=100)
cb_log.fit(X_bc_train, log_prices)
print(f"CatBoost train: {time.time()-t0:.1f}s")
results["CatBoost (B+Cat, log)"] = evaluate_log_model(cb_log, X_bc_test, test, "CatBoost (B+Cat, log)")

In [ ]:
# LightGBM (C+Cat, log)
t0 = time.time()
pbar_lgb2 = tqdm(total=1000, desc="LightGBM C")
def lgb_cb2(env): pbar_lgb2.update(1)
lgb_c_log = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=SEED, n_jobs=6, verbose=-1)
lgb_c_log.fit(X_cc_train, log_prices, callbacks=[lgb_cb2])
pbar_lgb2.close()
print(f"LightGBM C train: {time.time()-t0:.1f}s")
results["LightGBM (C+Cat, log)"] = evaluate_log_model(lgb_c_log, X_cc_test, test, "LightGBM (C+Cat, log)")

---## Phase 3: LightGBM Optuna Tuning

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 50, 200),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 0.7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.01, 1.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.01, 1.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": 1500, "random_state": SEED, "n_jobs": 6, "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_bc_train, log_prices)
    pred = np.clip(np.expm1(model.predict(X_bc_val)), 0, None)
    return rmsle(np.array(val_prices, dtype=float), pred)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f"Best val RMSLE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Train best on full train, eval on test
best_params = {**study.best_params, "n_estimators": 1500, "random_state": SEED, "n_jobs": 6, "verbose": -1}
lgb_tuned = lgb.LGBMRegressor(**best_params)
lgb_tuned.fit(X_bc_train, log_prices)
results["LightGBM Tuned (B+Cat, log)"] = evaluate_log_model(lgb_tuned, X_bc_test, test, "LightGBM Tuned (B+Cat, log)")

# Also Arch C
lgb_tuned_c = lgb.LGBMRegressor(**best_params)
lgb_tuned_c.fit(X_cc_train, log_prices)
results["LightGBM Tuned (C+Cat, log)"] = evaluate_log_model(lgb_tuned_c, X_cc_test, test, "LightGBM Tuned (C+Cat, log)")

---## Phase 4: Weighted Blending

In [ ]:
from scipy.optimize import minimize as scipy_minimize

pred_lgb_val = np.clip(np.expm1(lgb_tuned.predict(X_bc_val)), 0, None)
pred_xgb_val = np.clip(np.expm1(xgb_log.predict(X_bc_val)), 0, None)
pred_cb_val = np.clip(np.expm1(cb_log.predict(X_bc_val)), 0, None)
pred_lgb_c_val = np.clip(np.expm1(lgb_tuned_c.predict(X_cc_val)), 0, None)

val_true = np.array(val_prices, dtype=float)
val_preds = [pred_lgb_val, pred_xgb_val, pred_cb_val, pred_lgb_c_val]
names_blend = ["LGB Tuned B", "XGB B", "CB B", "LGB Tuned C"]

def blend_rmsle(w):
    w = np.abs(w); w = w / w.sum()
    return rmsle(val_true, sum(wi * p for wi, p in zip(w, val_preds)))

res_opt = scipy_minimize(blend_rmsle, x0=[0.4, 0.2, 0.2, 0.2], method="Nelder-Mead")
bw = np.abs(res_opt.x); bw = bw / bw.sum()
for n, w in zip(names_blend, bw): print(f"  {n}: {w:.3f}")
print(f"Val blend RMSLE: {res_opt.fun:.4f}")

# Apply to test
test_preds = [
    np.clip(np.expm1(lgb_tuned.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(xgb_log.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(cb_log.predict(X_bc_test)), 0, None),
    np.clip(np.expm1(lgb_tuned_c.predict(X_cc_test)), 0, None),
]
blended = sum(w * p for w, p in zip(bw, test_preds))
test_true = np.array([item.price for item in test], dtype=float)
from sklearn.metrics import r2_score
b_rmsle = rmsle(test_true, blended)
b_mae = float(np.mean(np.abs(test_true - blended)))
mask = test_true > 0
b_mape = float(np.mean(np.abs((test_true[mask] - blended[mask]) / test_true[mask])) * 100)
b_r2 = r2_score(test_true, blended) * 100
print(f"
Blended Test: RMSLE={b_rmsle:.4f} | MAE={b_mae:,.0f} | MAPE={b_mape:.1f}% | R2={b_r2:.1f}%")
results["Blended (4 models)"] = {"rmsle": b_rmsle, "mae": b_mae, "mape": b_mape, "r2": b_r2}

---## Final Summary

In [ ]:
import pandas as pd
summary = pd.DataFrame([{"Model": k, **v} for k, v in results.items()])
summary = summary.sort_values("rmsle")
print(summary.to_string(index=False))

best = min(results, key=lambda k: results[k]["rmsle"])
print(f"
Best: {best} -- RMSLE={results[best]['rmsle']:.4f}")
print(f"v2 best: 0.5799 | v3 best: {results[best]['rmsle']:.4f} | Improvement: {(0.5799-results[best]['rmsle'])/0.5799*100:.1f}%")